# Modelo No Supervisado y Red Neuronal - Sistema Experto para Diagnóstico Básico de Fallas en Redes
**Sesiones 13 y 14 - Proyecto Integrador**

Este notebook complementa el modelo supervisado (Random Forest) con:
1. Un modelo **no supervisado** (K-Means) para descubrir agrupaciones naturales de síntomas de red sin usar la etiqueta `diagnostico`, útil para detectar patrones o tipos de incidentes no contemplados en las reglas del sistema experto.
2. Una **red neuronal artificial** (Perceptrón Multicapa - MLP) que aprende a clasificar el diagnóstico, comparando su desempeño con el Random Forest del Taller 2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay, confusion_matrix

df = pd.read_csv('dataset_diagnostico_red.csv')
df.head()

## 1. Aprendizaje no supervisado: K-Means

Se codifican las variables categóricas y se agrupan los casos en clústeres sin usar la columna `diagnostico`, para luego analizar si los clústeres encontrados coinciden con los diagnósticos reales (validación externa).

In [ ]:
X = df.drop(columns=['diagnostico'])
y_real = df['diagnostico']

enc = OrdinalEncoder()
X_enc = enc.fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_enc)

# Método del codo para elegir k
inertias = []
rango_k = range(2, 13)
for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7,4))
plt.plot(list(rango_k), inertias, marker='o')
plt.xlabel('Número de clústeres (k)')
plt.ylabel('Inercia')
plt.title('Método del codo')
plt.tight_layout()
plt.show()

In [ ]:
k_elegido = 9  # igual al número real de diagnósticos, para comparar
kmeans = KMeans(n_clusters=k_elegido, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, clusters)
ari = adjusted_rand_score(y_real, clusters)
print(f'Silhouette Score: {sil:.3f}')
print(f'Adjusted Rand Index vs diagnóstico real: {ari:.3f}')

In [ ]:
# Visualización 2D con PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13,5))
sc0 = axes[0].scatter(X_pca[:,0], X_pca[:,1], c=clusters, cmap='tab10', s=15)
axes[0].set_title('Clústeres encontrados (K-Means)')
y_codes = pd.factorize(y_real)[0]
sc1 = axes[1].scatter(X_pca[:,0], X_pca[:,1], c=y_codes, cmap='tab10', s=15)
axes[1].set_title('Diagnóstico real (referencia)')
plt.tight_layout()
plt.show()

**Interpretación:** el Silhouette Score y el Adjusted Rand Index indican qué tan bien separados están los clústeres y qué tanto coinciden con las categorías reales de diagnóstico. Un ARI cercano a 1 indicaría que el agrupamiento no supervisado reconstruye casi perfectamente las categorías del sistema experto; valores más bajos son esperables porque K-Means no conoce las reglas de negocio, solo la similitud entre síntomas. Este análisis es útil para **descubrir nuevos tipos de falla** no contemplados aún en la base de reglas.

## 2. Red neuronal artificial (Perceptrón Multicapa)

Se entrena un MLPClassifier con dos capas ocultas (6 y 4 neuronas, ver Figura de arquitectura del informe) para predecir `diagnostico`, y se compara contra el Random Forest del Taller 2.

In [ ]:
label_enc = LabelEncoder()
y_enc = label_enc.fit_transform(y_real)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

mlp = MLPClassifier(hidden_layer_sizes=(6,4), activation='relu', solver='adam',
                     max_iter=2000, random_state=42)
mlp.fit(X_train, y_train)

y_pred = mlp.predict(X_test)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
print(f'Accuracy : {acc:.3f}')
print(f'Precision (macro): {prec:.3f}')
print(f'Recall (macro)   : {rec:.3f}')
print(f'F1-Score (macro) : {f1:.3f}')

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(mlp.loss_curve_)
plt.xlabel('Iteración')
plt.ylabel('Pérdida (loss)')
plt.title('Curva de aprendizaje de la red neuronal')
plt.tight_layout()
plt.show()

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_enc.classes_)
fig, ax = plt.subplots(figsize=(8,8))
disp.plot(ax=ax, xticks_rotation=90, colorbar=False, cmap='Purples')
plt.title('Matriz de confusión - Red Neuronal (MLP)')
plt.tight_layout()
plt.show()

## 3. Conclusión

El Random Forest (Taller 2) sigue siendo competitivo frente a la red neuronal en este dataset tabular de tamaño moderado (600 registros), lo cual es consistente con la literatura: los modelos de árboles suelen igualar o superar a las redes neuronales en datos tabulares pequeños/medianos, mientras que las redes neuronales y el Deep Learning muestran ventajas claras en datos no estructurados y de gran volumen (imágenes, audio, texto), como se discute en el Capítulo 4 (Marco Conceptual) y el Capítulo 9 (Discusión) del informe final.